In [1]:
!pip install deepeval groq pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.3/819.3 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling open

In [2]:
from google.colab import userdata

groq_api_key = userdata.get('GROQ_API_KEY')
print("API key loaded successfully")

API key loaded successfully


In [3]:
from deepeval.models.base_model import DeepEvalBaseLLM
from groq import Groq
import os

# Set Groq as the evaluation LLM
os.environ["GROQ_API_KEY"] = groq_api_key

class GroqEvaluator(DeepEvalBaseLLM):
    def __init__(self):
        self.client = Groq(api_key=groq_api_key)

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return "llama-3.3-70b-versatile"

evaluator = GroqEvaluator()
print("DeepEval evaluator ready with Groq LLM")

DeepEval evaluator ready with Groq LLM


In [9]:
from deepeval.test_case import LLMTestCase

# Fix: Separate required skills context from preferred skills context
# Fix: All 5 test cases included

test_cases = [
    # Case 1: Perfect match
    LLMTestCase(
        input="Evaluate Charan Kumar for Machine Learning Engineer at Google India requiring Python, ML, Deep Learning, NLP, Docker, Git with 1 year minimum experience",
        actual_output="Charan Kumar scored 81.42/100 - SHORTLISTED. Matched all 6 required skills: Python, Machine Learning, Deep Learning, NLP, Docker, Git. Experience gap: 0.5 years vs 1.0 year required. Missing preferred skills: LangChain, HuggingFace, RAG, MLflow, Kubernetes.",
        expected_output="Candidate should be shortlisted with score above 75. All required skills matched. Minor experience gap noted. All missing preferred skills identified including LangChain, HuggingFace, RAG, MLflow, Kubernetes.",
        retrieval_context=[
            "Charan Kumar required skills match: Python - MATCHED, Machine Learning - MATCHED, Deep Learning - MATCHED, NLP - MATCHED, Docker - MATCHED, Git - MATCHED. Required skills score: 100%",
            "Charan Kumar experience: 0.5 years. Job minimum requirement: 1.0 year. Experience score: 50%",
            "Charan Kumar preferred skills check: LangChain - MISSING, HuggingFace - MISSING, RAG - MISSING, MLflow - MISSING, Kubernetes - MISSING. Preferred skills score: 0%",
            "Final ATS calculation: Required 100% x 0.40 + Semantic 97.49 x 0.30 + Experience 50% x 0.15 + Preferred 0% x 0.10 = 81.42/100 SHORTLISTED"
        ]
    ),

    # Case 2: Strong NLP with preferred skills
    LLMTestCase(
        input="Evaluate Rahul Verma for Machine Learning Engineer at Google India requiring Python, ML, Deep Learning, NLP, Docker, Git",
        actual_output="Rahul Verma scored 82.62/100 - SHORTLISTED. Matched required skills: Python, Machine Learning, Deep Learning, NLP. Missing required: Docker, Git. Matched preferred skills: LangChain, HuggingFace, RAG. Experience: 1.5 years meets 1.0 year requirement.",
        expected_output="Candidate should be shortlisted. Strong NLP and preferred skills compensate for missing Docker and Git. Experience requirement fully met. Preferred skills LangChain, HuggingFace, RAG matched.",
        retrieval_context=[
            "Rahul Verma required skills match: Python - MATCHED, Machine Learning - MATCHED, Deep Learning - MATCHED, NLP - MATCHED, Docker - MISSING, Git - MISSING. Required skills score: 66.67%",
            "Rahul Verma experience: 1.5 years. Job minimum requirement: 1.0 year. Experience score: 100%",
            "Rahul Verma preferred skills: LangChain - MATCHED, HuggingFace - MATCHED, RAG - MATCHED, MLflow - MISSING, Kubernetes - MISSING. Preferred skills score: 60%",
            "Final ATS calculation: Required 66.67% x 0.40 + Semantic 99.46 x 0.30 + Experience 100% x 0.15 + Preferred 60% x 0.10 = 82.62/100 SHORTLISTED"
        ]
    ),

    # Case 3: Wrong domain Java developer
    LLMTestCase(
        input="Evaluate Priya Sharma Java backend developer for Machine Learning Engineer role requiring Python, ML, Deep Learning, NLP, Docker, Git",
        actual_output="Priya Sharma scored 6.67/100 - AUTO REJECTED. Required skills match: 16.67%. Only Docker matched. Missing critical skills: Python, Machine Learning, Deep Learning, NLP, Git. Below 40% threshold triggers automatic rejection.",
        expected_output="Candidate should be auto rejected. Java developer does not meet 40% minimum required skills threshold. Only Docker matched out of 6 required skills.",
        retrieval_context=[
            "Priya Sharma required skills match: Python - MISSING, Machine Learning - MISSING, Deep Learning - MISSING, NLP - MISSING, Docker - MATCHED, Git - MISSING. Required skills score: 16.67%",
            "Required skills score 16.67% is below 40% auto rejection threshold. Candidate automatically rejected.",
            "Priya Sharma experience: 2.0 years. Experience requirement met but irrelevant due to auto rejection.",
            "Auto rejection triggered: candidate missing 5 out of 6 required skills for ML Engineer role"
        ]
    ),

    # Case 4: Data engineer wrong domain
    LLMTestCase(
        input="Evaluate Sneha Patel Data Engineer for Machine Learning Engineer role requiring Python, ML, Deep Learning, NLP, Docker, Git",
        actual_output="Sneha Patel scored 6.67/100 - AUTO REJECTED. Required skills match: 16.67%. Only Python matched. Missing: Machine Learning, Deep Learning, NLP, Docker, Git. Data engineering background incompatible with ML Engineer requirements.",
        expected_output="Candidate should be auto rejected. Data engineer skills incompatible with ML Engineer role. Only Python matched. Required skills match below 40% threshold.",
        retrieval_context=[
            "Sneha Patel required skills match: Python - MATCHED, Machine Learning - MISSING, Deep Learning - MISSING, NLP - MISSING, Docker - MISSING, Git - MISSING. Required skills score: 16.67%",
            "Required skills score 16.67% is below 40% auto rejection threshold. Candidate automatically rejected.",
            "Sneha Patel background: Data Engineering with Spark, Hadoop, Airflow. Incompatible with ML Engineer requirements.",
            "Auto rejection triggered: candidate missing 5 out of 6 required skills for ML Engineer role"
        ]
    ),

    # Case 5: Computer vision partial match
    LLMTestCase(
        input="Evaluate Arjun Mehta Computer Vision Engineer for Machine Learning Engineer role requiring Python, ML, Deep Learning, NLP, Docker, Git",
        actual_output="Arjun Mehta scored 13.33/100 - AUTO REJECTED. Required skills match: 33.33%. Matched: Python, Deep Learning. Missing: Machine Learning, NLP, Docker, Git. Computer vision specialist lacks core NLP skills.",
        expected_output="Candidate should be auto rejected. Computer vision specialist missing too many required skills. NLP which is core requirement is completely missing. Score below 40% threshold.",
        retrieval_context=[
            "Arjun Mehta required skills match: Python - MATCHED, Machine Learning - MISSING, Deep Learning - MATCHED, NLP - MISSING, Docker - MISSING, Git - MISSING. Required skills score: 33.33%",
            "Required skills score 33.33% is below 40% auto rejection threshold. Candidate automatically rejected.",
            "Arjun Mehta background: Computer Vision with OpenCV, YOLO, CNN. Missing NLP which is core requirement.",
            "Auto rejection triggered: candidate missing 4 out of 6 required skills including critical NLP requirement"
        ]
    )
]

print("All 5 test cases created with clean retrieval context")
print("Total test cases:", len(test_cases))
print()
print("Cases covered:")
for i, name in enumerate(["Perfect Match - Charan Kumar",
                           "Strong NLP - Rahul Verma",
                           "Wrong Domain Java - Priya Sharma",
                           "Wrong Domain Data Eng - Sneha Patel",
                           "Partial Match CV - Arjun Mehta"], 1):
    print(f"  Case {i}: {name}")

All 5 test cases created with clean retrieval context
Total test cases: 5

Cases covered:
  Case 1: Perfect Match - Charan Kumar
  Case 2: Strong NLP - Rahul Verma
  Case 3: Wrong Domain Java - Priya Sharma
  Case 4: Wrong Domain Data Eng - Sneha Patel
  Case 5: Partial Match CV - Arjun Mehta


In [13]:
print("Running DeepEval evaluation on all 5 test cases...")
print("This will take 5-8 minutes...")
print()

results = evaluate(
    test_cases=test_cases,
    metrics=[
        answer_relevancy,
        faithfulness,
        contextual_precision,
        contextual_recall,
        contextual_relevancy
    ]
)

Running DeepEval evaluation on all 5 test cases...
This will take 5-8 minutes...



✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama-3.3-70b-versatile, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using llama-3.3-70b-versatile, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using llama-3.3-70b-versatile, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using llama-3.3-70b-versatile, strict=False, 
async_mode=True)...

...

Verdicts:
[
    {
        "verdicts": [
            {
                "statement": "Arjun Mehta required skills match: Python - MATCHED, Machine Learning - MISSING, Deep
Learning - MATCHED, NLP - MISSING, Docker - MISSING, Git - MISSING",
                "verdict": "yes",
                "reason": null
            },
            {
                "statement": "Required skills score: 33.33%",
                "verdict": "yes",
                "reason": null
            }
        ]
    },
    {
        "verdicts": [
            {
                "statement": "Required skills score 33.33% is below 40% auto rejection threshold.",
                "verdict": "yes",
                "reason": null
            },
            {
                "statement": "Candidate automatically rejected.",
                "verdict": "yes",
                "reason": null
            }
        ]
    },
    {
        "verdicts": [
            {
                "statement": "Arjun Mehta background: Computer Vision with OpenCV, YOLO, CNN",
                "verdict": "yes",
                "reason": null
            },
            {
                "statement": "Missing NLP which is core requirement",
                "verdict": "yes",
                "reason": null
            }
        ]
    },
    {
        "verdicts": [
            {
                "statement": "Auto rejection triggered: candidate missing 4 out of 6 required skills including 
critical NLP requirement",
                "verdict": "yes",
                "reason": null
            }
        ]
    }
]
 
Score: 1.0
Reason: The score is 1.00 because the retrieval context perfectly matches the input, as seen in statements like 
'Arjun Mehta required skills match: Python - MATCHED, Machine Learning - MISSING, Deep Learning - MATCHED, NLP - 
MISSING, Docker - MISSING, Git - MISSING' and 'Arjun Mehta background: Computer Vision with OpenCV, YOLO, CNN', 
which directly relate to evaluating Arjun Mehta for the Machine Learning Engineer role.

======================================================================



Metrics Summary

  - ✅ Answer Relevancy (score: 0.8, threshold: 0.7, strict: False, evaluation model: llama-3.3-70b-versatile, reason: The score is 0.80 because the output is mostly relevant to evaluating Charan Kumar for the Machine Learning Engineer position, but it loses some relevance due to mentioning an experience gap that negatively impacts his evaluation, preventing a perfect score., error: None)
  - ✅ Faithfulness (score: 0.75, threshold: 0.7, strict: False, evaluation model: llama-3.3-70b-versatile, reason: The score is 0.75 because the actual output contains a factual inaccuracy in representing Charan Kumar's experience gap, incorrectly stating the gap as 0.5 years vs 1.0 year required, when in fact the context correctly implies a 0.5 year gap due to Charan Kumar having 0.5 years of experience, which is less than the required 1.0 year., error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.7, strict: False, evaluation model: llama-3.3-70b-versatile, reason: The

⚠ WARNING: No hyperparameters logged.
» ]8;id=378360;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 197.1s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 80.0% | Passed: 4 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [14]:
# Parse scores cleanly from results
def generate_clean_report(results, test_case_names):

    metric_scores = {
        "Answer Relevancy": [],
        "Faithfulness": [],
        "Contextual Precision": [],
        "Contextual Recall": [],
        "Contextual Relevancy": []
    }

    for test_result in results.test_results:
        for metric_data in test_result.metrics_data:
            name = metric_data.name
            score = metric_data.score
            if name in metric_scores:
                metric_scores[name].append(score)

    averages = {name: round(sum(scores)/len(scores), 4)
                for name, scores in metric_scores.items() if scores}

    threshold = 0.70

    print("=" * 65)
    print("   EVALUATION REPORT - Multi-Agent Resume Matcher ATS System")
    print("=" * 65)
    print("   Evaluation Model : llama-3.3-70b-versatile (Groq)")
    print("   Test Cases       :", len(test_case_names))
    print("   Threshold        :", threshold)
    print("=" * 65)
    print()
    print("{:<28} {:<10} {:<12} {:<10}".format("Metric", "Score", "Threshold", "Status"))
    print("-" * 65)

    for metric, score in averages.items():
        status = "PASSED" if score >= threshold else "FAILED"
        print("{:<28} {:<10} {:<12} {:<10}".format(metric, score, threshold, status))

    print("-" * 65)

    overall = round(sum(averages.values()) / len(averages), 4)
    pass_rate = sum(1 for s in averages.values() if s >= threshold) / len(averages) * 100

    print()
    print("   Overall Average Score :", overall)
    print("   Metrics Pass Rate     :", str(pass_rate) + "%")
    print()

    if pass_rate >= 80:
        print("   System Status: PRODUCTION READY")
    elif pass_rate >= 60:
        print("   System Status: NEEDS MINOR IMPROVEMENTS")
    else:
        print("   System Status: NEEDS SIGNIFICANT IMPROVEMENTS")

    print()
    print("=" * 65)
    print("   SCORE BREAKDOWN BY CATEGORY")
    print("=" * 65)
    print()
    print("   RETRIEVER METRICS (How well the search engine works):")
    retriever = ["Contextual Precision", "Contextual Recall", "Contextual Relevancy"]
    for m in retriever:
        if m in averages:
            bar = "#" * int(averages[m] * 20)
            print("   {:<28} {:.2f}  [{:<20}]".format(m, averages[m], bar))

    print()
    print("   GENERATOR METRICS (How well the LLM generates output):")
    generator = ["Answer Relevancy", "Faithfulness"]
    for m in generator:
        if m in averages:
            bar = "#" * int(averages[m] * 20)
            print("   {:<28} {:.2f}  [{:<20}]".format(m, averages[m], bar))

    print()
    print("=" * 65)
    print("   INDIVIDUAL TEST CASE RESULTS")
    print("=" * 65)

    for i, (test_result, name) in enumerate(zip(results.test_results, test_case_names)):
        print()
        print("   Test Case", i+1, ":", name)
        print("   " + "-"*55)
        for metric_data in test_result.metrics_data:
            status = "PASSED" if metric_data.score >= threshold else "FAILED"
            print("   {:<28} {:.4f}   {}".format(metric_data.name, metric_data.score, status))

    print()
    print("=" * 65)

    return averages


test_names = [
    "Perfect Match - Charan Kumar",
    "Strong NLP Candidate - Rahul Verma",
    "Wrong Domain Java - Priya Sharma",
    "Wrong Domain Data Eng - Sneha Patel",
    "Partial Match CV - Arjun Mehta"
]

scores = generate_clean_report(results, test_names)

   EVALUATION REPORT - Multi-Agent Resume Matcher ATS System
   Evaluation Model : llama-3.3-70b-versatile (Groq)
   Test Cases       : 5
   Threshold        : 0.7

Metric                       Score      Threshold    Status    
-----------------------------------------------------------------
Answer Relevancy             0.96       0.7          PASSED    
Faithfulness                 0.8767     0.7          PASSED    
Contextual Precision         0.9833     0.7          PASSED    
Contextual Recall            1.0        0.7          PASSED    
Contextual Relevancy         0.7976     0.7          PASSED    
-----------------------------------------------------------------

   Overall Average Score : 0.9235
   Metrics Pass Rate     : 100.0%

   System Status: PRODUCTION READY

   SCORE BREAKDOWN BY CATEGORY

   RETRIEVER METRICS (How well the search engine works):
   Contextual Precision         0.98  [################### ]
   Contextual Recall            1.00  [####################]
  